In [0]:
-- Question 3 manual trace: Bronze -> Silver -> Gold
--
-- series_id = PRS30006032
-- period    = Q01
--
-- Test 1: 2018 -> BLS + population should both exist
-- Test 2: 1994 -> BLS exists; population should be NULL
-- Test 3: 2026 -> BLS exists; population should be NULL
-- Test 4: population year with no matching BLS Q01 -> diagnostic; ideally 0 rows


-- ============================================================
-- TEST 1-3: BRONZE BLS
-- Expected BLS values from pr.data.1.AllData:
--   1994 = 1.7
--   2018 = 0.5
--   2026 = 0.0
-- ============================================================

WITH test_cases AS (
    SELECT * FROM VALUES
        (1, 'BLS + population',     2018),
        (2, 'BLS, no population',   1994),
        (3, 'Recent BLS, no population', 2026)
    AS t(test_case, test_name, year)
)

SELECT
    t.test_case,
    t.test_name,
    b.series_id,
    CAST(b.year AS INT) AS year,
    b.period,
    CAST(b.value AS DOUBLE) AS bronze_bls_value
FROM rearc.bronze.bls_pr_data_1_alldata b
JOIN test_cases t
    ON CAST(b.year AS INT) = t.year
WHERE b.series_id = 'PRS30006032'
  AND b.period = 'Q01'
ORDER BY t.test_case;


-- ============================================================
-- TEST 1-3: BRONZE POPULATION
-- Use only the newest raw population snapshot
-- ============================================================

WITH test_cases AS (
    SELECT * FROM VALUES
        (1, 'BLS + population',     2018),
        (2, 'BLS, no population',   1994),
        (3, 'Recent BLS, no population', 2026)
    AS t(test_case, test_name, year)
),

latest_population_snapshot AS (
    SELECT raw_json
    FROM rearc.bronze.population_raw
    ORDER BY fetched_at_utc DESC
    LIMIT 1
),

population AS (
    SELECT
        CAST(p.Year AS INT) AS year,
        CAST(p.Population AS BIGINT) AS population
    FROM latest_population_snapshot
    LATERAL VIEW EXPLODE(
        FROM_JSON(
            GET_JSON_OBJECT(raw_json, '$.data'),
            'ARRAY<STRUCT<`Nation ID`:STRING, Nation:STRING, Year:BIGINT, Population:DOUBLE>>'
        )
    ) exploded AS p
)

SELECT
    t.test_case,
    t.test_name,
    t.year,
    p.population AS bronze_population
FROM test_cases t
LEFT JOIN population p
    ON t.year = p.year
ORDER BY t.test_case;


-- ============================================================
-- TEST 1-3: SILVER BLS
-- Values should match Bronze BLS exactly
-- ============================================================

WITH test_cases AS (
    SELECT * FROM VALUES
        (1, 'BLS + population',     2018),
        (2, 'BLS, no population',   1994),
        (3, 'Recent BLS, no population', 2026)
    AS t(test_case, test_name, year)
)

SELECT
    t.test_case,
    t.test_name,
    s.series_id,
    s.year,
    s.period,
    s.value AS silver_bls_value
FROM rearc.silver.fct_bls_pr_data_1_alldata s
JOIN test_cases t
    ON s.year = t.year
WHERE s.series_id = 'PRS30006032'
  AND s.period = 'Q01'
ORDER BY t.test_case;


-- ============================================================
-- TEST 1-3: GENERIC GOLD
-- All series + periods
-- ============================================================

WITH test_cases AS (
    SELECT * FROM VALUES
        (1, 'BLS + population',     2018),
        (2, 'BLS, no population',   1994),
        (3, 'Recent BLS, no population', 2026)
    AS t(test_case, test_name, year)
)

SELECT
    t.test_case,
    t.test_name,
    g.series_id,
    g.year,
    g.period,
    g.value,
    g.population
FROM rearc.gold.bls_series_period_population_by_year g
JOIN test_cases t
    ON g.year = t.year
WHERE g.series_id = 'PRS30006032'
  AND g.period = 'Q01'
ORDER BY t.test_case;


-- ============================================================
-- TEST 1-3: QUESTION-SPECIFIC GOLD
-- ============================================================

WITH test_cases AS (
    SELECT * FROM VALUES
        (1, 'BLS + population',     2018),
        (2, 'BLS, no population',   1994),
        (3, 'Recent BLS, no population', 2026)
    AS t(test_case, test_name, year)
)

SELECT
    t.test_case,
    t.test_name,
    g.*
FROM rearc.gold.bls_prs30006032_q01_population_by_year g
JOIN test_cases t
    ON g.year = t.year
ORDER BY t.test_case;


-- ============================================================
-- TEST 4: POPULATION YEAR WITH NO MATCHING BLS Q01
-- This identifies the case if one exists in the current data.
-- ============================================================

WITH latest_population_snapshot AS (
    SELECT raw_json
    FROM rearc.bronze.population_raw
    ORDER BY fetched_at_utc DESC
    LIMIT 1
),

population AS (
    SELECT
        CAST(p.Year AS INT) AS year,
        CAST(p.Population AS BIGINT) AS population
    FROM latest_population_snapshot
    LATERAL VIEW EXPLODE(
        FROM_JSON(
            GET_JSON_OBJECT(raw_json, '$.data'),
            'ARRAY<STRUCT<`Nation ID`:STRING, Nation:STRING, Year:BIGINT, Population:DOUBLE>>'
        )
    ) exploded AS p
),

bls AS (
    SELECT
        year,
        value
    FROM rearc.silver.fct_bls_pr_data_1_alldata
    WHERE series_id = 'PRS30006032'
      AND period = 'Q01'
)

SELECT
    4 AS test_case,
    'Population, no BLS Q01' AS test_name,
    p.year,
    p.population
FROM population p
LEFT JOIN bls b
    ON p.year = b.year
WHERE b.year IS NULL
ORDER BY p.year;


-- ============================================================
-- FINAL COMPARISON: BRONZE vs SILVER vs BOTH GOLD TABLES
-- A difference of 0 means the values agree.
-- ============================================================

WITH test_cases AS (
    SELECT * FROM VALUES
        (1, 'BLS + population',     2018),
        (2, 'BLS, no population',   1994),
        (3, 'Recent BLS, no population', 2026)
    AS t(test_case, test_name, year)
),

bronze_bls AS (
    SELECT
        CAST(year AS INT) AS year,
        CAST(value AS DOUBLE) AS value
    FROM rearc.bronze.bls_pr_data_1_alldata
    WHERE series_id = 'PRS30006032'
      AND period = 'Q01'
),

latest_population_snapshot AS (
    SELECT raw_json
    FROM rearc.bronze.population_raw
    ORDER BY fetched_at_utc DESC
    LIMIT 1
),

bronze_population AS (
    SELECT
        CAST(p.Year AS INT) AS year,
        CAST(p.Population AS BIGINT) AS population
    FROM latest_population_snapshot
    LATERAL VIEW EXPLODE(
        FROM_JSON(
            GET_JSON_OBJECT(raw_json, '$.data'),
            'ARRAY<STRUCT<`Nation ID`:STRING, Nation:STRING, Year:BIGINT, Population:DOUBLE>>'
        )
    ) exploded AS p
),

silver_bls AS (
    SELECT
        year,
        value
    FROM rearc.silver.fct_bls_pr_data_1_alldata
    WHERE series_id = 'PRS30006032'
      AND period = 'Q01'
),

generic_gold AS (
    SELECT
        year,
        value,
        population
    FROM rearc.gold.bls_series_period_population_by_year
    WHERE series_id = 'PRS30006032'
      AND period = 'Q01'
),

specific_gold AS (
    SELECT
        year,
        value,
        population
    FROM rearc.gold.bls_prs30006032_q01_population_by_year
)

SELECT
    t.test_case,
    t.test_name,
    t.year,

    bb.value AS bronze_bls_value,
    sb.value AS silver_bls_value,
    gg.value AS generic_table_gold_value,
    sg.value AS specific_table_gold_value,

    bp.population AS bronze_population,
    gg.population AS generic_gold_population,
    sg.population AS specific_gold_population,

    ABS(bb.value - sb.value) AS bronze_vs_silver_value_diff,
    ABS(sb.value - gg.value) AS silver_vs_generic_table_gold_value_diff,
    ABS(gg.value - sg.value) AS generic_vs_specific_table_gold_value_diff,

    CASE
        WHEN bp.population IS NULL AND gg.population IS NULL THEN 0
        ELSE ABS(bp.population - gg.population)
    END AS bronze_vs_generic_gold_population_diff,

    CASE
        WHEN gg.population IS NULL AND sg.population IS NULL THEN 0
        ELSE ABS(gg.population - sg.population)
    END AS generic_vs_specific_gold_population_diff

FROM test_cases t
LEFT JOIN bronze_bls bb
    ON t.year = bb.year
LEFT JOIN bronze_population bp
    ON t.year = bp.year
LEFT JOIN silver_bls sb
    ON t.year = sb.year
LEFT JOIN generic_gold gg
    ON t.year = gg.year
LEFT JOIN specific_gold sg
    ON t.year = sg.year

ORDER BY t.test_case;
